# Bagh-level cross-scale validation

This notebook evaluates the 1-km livestock spatialization product against official Mongolian bagh statistics for 2012–2023. It focuses on two quantities that directly describe within-soum allocation: bagh shares of the soum total and the rank ordering of baghs within each soum–year.


## 1. Read the finalized validation tables

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
from scipy.stats import linregress, spearmanr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

VALIDATION_DIR = Path(r"E:\MPLD\Bagh\run\04_validation")
OUTPUT_DIR = Path(r"E:\MPLD\Figure")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELED_TABLE = VALIDATION_DIR / "bagh_modeled_SU_2012_2023.csv"
OFFICIAL_TABLE = VALIDATION_DIR / "bagh_official_SU_2012_2023.csv"

modeled = pd.read_csv(MODELED_TABLE)
official = pd.read_csv(OFFICIAL_TABLE)
keys = ["STAT_ID", "year", "PARENT_ID"]
data = official.merge(modeled[keys + ["modeled_bagh_SU", "modeled_soum_SU", "modeled_share"]],
    on=keys,how="inner",validate="one_to_one",)

print(f"Bagh-year records: {len(data):,}")
print(f"Unique baghs: {data['STAT_ID'].nunique():,}")
print(f"Unique soums: {data['PARENT_ID'].nunique():,}")
print(f"Soum-years: {data[['PARENT_ID', 'year']].drop_duplicates().shape[0]:,}")


## 2. Within-soum allocation

The official and spatialized livestock abundance of each bagh is expressed as a share of its parent soum total. The reported $R^2$ is the squared Pearson correlation from a linear fit; MAE is also expressed in percentage points for interpretation.


In [ ]:
x_share = data["official_share"].to_numpy(float)
y_share = data["modeled_share"].to_numpy(float)
fit = linregress(x_share, y_share)

allocation_metrics = pd.Series({
    "R2": fit.rvalue ** 2,
    "Spearman_rho": spearmanr(x_share, y_share).statistic,
    "RMSE": np.sqrt(np.mean((y_share - x_share) ** 2)),
    "MAE": np.mean(np.abs(y_share - x_share)),
    "MAE_percentage_points": np.mean(np.abs(y_share - x_share)) * 100,
})
allocation_metrics


## 3. Within-soum ranking

For every soum–year containing at least three valid baghs, Spearman's $\rho$ is calculated between official and spatialized bagh livestock abundance.


In [ ]:
rank_rows = []
for (parent_id, year), group in data.groupby(["PARENT_ID", "year"], sort=True):
    if len(group) < 3:
        continue
    if group["official_bagh_SU"].nunique() < 2 or group["modeled_bagh_SU"].nunique() < 2:
        continue
    rho = spearmanr(group["official_bagh_SU"], group["modeled_bagh_SU"]).statistic
    if np.isfinite(rho):
        rank_rows.append({
            "PARENT_ID": int(parent_id),
            "year": int(year),
            "n_bagh": len(group),
            "rho": float(rho),
        })

ranks = pd.DataFrame(rank_rows)
rank_metrics = pd.Series({
    "n_soum_year": len(ranks),
    "median_rho": ranks["rho"].median(),
    "rho_gt_0_percent": (ranks["rho"] > 0).mean() * 100,
    "rho_gt_0.5_percent": (ranks["rho"] > 0.5).mean() * 100,
})
rank_metrics


## 4. Final 1.5-column validation figure

In [ ]:
# ==================== 1. Figure geometry: mm ====================
LEFT = 10.0      # 左留白：Figure左边缘到(a)黑框左边，mm
RIGHT = 2.0      # 右留白：(b)黑框右边到Figure右边缘，mm
BOTTOM = 8.0    # 下留白：黑框底边到Figure底边，mm
TOP = 5.0       # 上留白：黑框顶边到Figure顶边，mm
AX_W = 50.0      # a、b黑框宽度，mm
AX_H = 50.0      # a、b黑框高度，mm
GAP = 15.0       # a、b黑框之间净距离，mm

FIG_W = LEFT + AX_W * 2 + GAP + RIGHT
FIG_H = BOTTOM + AX_H + TOP

TITLE_Y = 1.01  # 标题纵向位置，1.0=黑框顶边
XLABEL_Y = -0.1 # X轴标题纵向位置，0=黑框底边
YLABEL_X = -0.12 # Y轴标题横向位置，0=黑框左边

def axes_rect_mm(left_mm, bottom_mm, width_mm, height_mm):
    return [left_mm / FIG_W, bottom_mm / FIG_H, width_mm / FIG_W, height_mm / FIG_H]

# ==================== 2. Point density ====================
def smoothed_point_density(x, y, bins=180, sigma=1.55):
    hist, x_edges, y_edges = np.histogram2d(x, y, bins=bins, range=((0, 1), (0, 1)))
    smooth = gaussian_filter(hist, sigma=sigma, mode="nearest")
    ix = np.clip(np.searchsorted(x_edges, x, side="right") - 1, 0, bins - 1)
    iy = np.clip(np.searchsorted(y_edges, y, side="right") - 1, 0, bins - 1)
    density = np.log1p(smooth[ix, iy])
    low, high = np.percentile(density, [1, 99.5])
    if high <= low:
        return np.zeros_like(density)
    return np.clip((density - low) / (high - low), 0, 1)

density = smoothed_point_density(x_share, y_share)
draw_order = np.argsort(density)
viridis = mpl.colormaps["viridis"]
muted_viridis = LinearSegmentedColormap.from_list("muted_viridis", viridis(np.linspace(0.12, 0.74, 256)))

# ==================== 3. Global style ====================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

blue = "#2369A0"
orange = "#D9772B"
light_gray = "#C7C7C7"
dark = "#3A3A3A"
text_backdrop = {"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "none", "alpha": 0.70}

def four_spines(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)

# ==================== 4. Create figure and exact axes ====================
fig = plt.figure(figsize=(FIG_W / 25.4, FIG_H / 25.4), constrained_layout=False)
ax_a = fig.add_axes(axes_rect_mm(LEFT, BOTTOM, AX_W, AX_H))
ax_b = fig.add_axes(axes_rect_mm(LEFT + AX_W + GAP, BOTTOM, AX_W, AX_H))
axes = [ax_a, ax_b]

# ==================== 5. Panel (a): Within-soum allocation ====================
ax = ax_a
ax.scatter(x_share[draw_order], y_share[draw_order], c=density[draw_order], s=5.0, cmap=muted_viridis, vmin=0, vmax=1, alpha=0.56, edgecolors="none", rasterized=True)
ax.plot([0, 1], [0, 1], "--", color=dark, lw=1.0, dashes=(4, 2))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Official bagh share of soum total")
ax.set_ylabel("Spatialized bagh share of soum total")
ax.set_aspect("equal", adjustable="box")
ax.set_title("(a) Within-soum allocation", loc="left", x=0.0, y=TITLE_Y, pad=0.2, fontsize=9, fontweight="normal")
ax.text(0.045, 0.955, f"$R^2$ = {allocation_metrics['R2']:.3f}\n$\\rho$ = {allocation_metrics['Spearman_rho']:.3f}\nMAE = {allocation_metrics['MAE_percentage_points']:.1f} percentage points\n$n$ = {len(data):,}", transform=ax.transAxes, va="top", 
        fontsize=8, linespacing=1.18, bbox=text_backdrop, zorder=8)
four_spines(ax)

# ==================== 6. Panel (b): Within-soum ranking ====================
ax = ax_b
rank_values = ranks["rho"].to_numpy(float)
rank_values = rank_values[np.isfinite(rank_values)]
sorted_rho = np.sort(rank_values)
ecdf = np.arange(1, len(sorted_rho) + 1) / len(sorted_rho)
plot_rho = np.r_[sorted_rho, 1.02]
plot_ecdf = np.r_[ecdf, 1.0]

ax.step(plot_rho, plot_ecdf, where="post", color=blue, lw=1.8)
ax.axvline(0, color=light_gray, lw=0.62, ls=(0, (3, 2)), alpha=0.85)
ax.axvline(0.5, color=light_gray, lw=0.62, ls=(0, (3, 2)), alpha=0.85)
ax.axvline(rank_metrics["median_rho"], color=orange, lw=1.35, ls=(0, (4, 2)))
ax.set_xlim(-1.02, 1.02)
ax.set_ylim(0, 1.01)
ax.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
ax.set_xlabel("Within-soum Spearman's $\\rho$")
ax.set_ylabel("Cumulative proportion of soum–years")
ax.set_aspect("auto")
ax.set_title("(b) Within-soum ranking", loc="left", x=0.0, y=TITLE_Y, pad=0.2, fontsize=9, fontweight="normal")
ax.text(0.045, 0.955, f"Median $\\rho$ = {rank_metrics['median_rho']:.3f}\n$\\rho$ > 0: {rank_metrics['rho_gt_0_percent']:.1f}%\n$\\rho$ > 0.5: {rank_metrics['rho_gt_0.5_percent']:.1f}%\n$n$ = {len(ranks):,}", 
        transform=ax.transAxes, va="top", fontsize=8, linespacing=1.18, bbox=text_backdrop, zorder=8)
four_spines(ax)

# ==================== 7. Unified alignment ====================
for ax in axes:
    ax.tick_params(axis="both", which="major", direction="out", pad=2.0)
    ax.xaxis.set_label_coords(0.5, XLABEL_Y)
    ax.yaxis.set_label_coords(YLABEL_X, 0.5)

# ==================== 8. Check physical dimensions ====================
fig.canvas.draw()
for name, ax in zip(["a", "b"], axes):
    bbox = ax.get_position()
    width_mm = bbox.width * FIG_W
    height_mm = bbox.height * FIG_H
    left_mm = bbox.x0 * FIG_W
    bottom_mm = bbox.y0 * FIG_H
    print(f"Panel ({name}): left={left_mm:.2f} mm, bottom={bottom_mm:.2f} mm, width={width_mm:.2f} mm, height={height_mm:.2f} mm")

bbox_a = ax_a.get_position()
bbox_b = ax_b.get_position()
actual_gap = (bbox_b.x0 - bbox_a.x1) * FIG_W
print(f"Figure size = {FIG_W:.2f} × {FIG_H:.2f} mm")
print(f"Gap between panels = {actual_gap:.2f} mm")

# ==================== 9. Save ====================
plt.savefig(OUTPUT_DIR / "Fig5_ab_bagh_validation.png", dpi=1200, bbox_inches=None, facecolor="white")
plt.savefig(OUTPUT_DIR / "Fig5_ab_bagh_validation.pdf", bbox_inches=None, facecolor="white")
#plt.savefig(OUTPUT_DIR / "Fig5_ab_bagh_validation.svg", bbox_inches=None, facecolor="white")

plt.show()


## 5. Spatial distribution of within-soum ranking consistency


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

# ==================== 1. Figure geometry: mm ====================
FIG_W = 127.0
FIG_H = 53.0

MAP_LEFT = 2.0
MAP_RIGHT_GAP = 1.0
LEG_W = 18.0
RIGHT = 1.0

TOP = 6.0
BOTTOM = 0.5

MAP_W = FIG_W - MAP_LEFT - MAP_RIGHT_GAP - LEG_W - RIGHT
MAP_H = FIG_H - TOP - BOTTOM
LEG_LEFT = MAP_LEFT + MAP_W + MAP_RIGHT_GAP
LEG_H = MAP_H

TITLE_LEFT = 10.0

def rect_mm(fig_w, fig_h, left_mm, bottom_mm, width_mm, height_mm):
    return [left_mm / fig_w, bottom_mm / fig_h, width_mm / fig_w, height_mm / fig_h]

# ==================== 2. Global style ====================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# ==================== 3. Paths ====================
SOUM_BOUNDARY = Path(r"E:\MPLD\SU\data\SU0_Official_442_county.shp")
REFERENCE_GRID = Path(r"E:\MPLD\geometry\M1_IM2_1km.tif")

# ==================== 4. Prepare soum-level ranking data ====================
soum_summary = ranks.groupby("PARENT_ID", as_index=False).agg(
    median_rho=("rho", "median"),
    n_valid_years=("year", "nunique")
)

soums = gpd.read_file(SOUM_BOUNDARY)
mongolia = soums.loc[soums["ADM0_NAME"].eq("Mongolia")].copy()

with rasterio.open(REFERENCE_GRID) as source:
    mongolia = mongolia.to_crs(source.crs)

mapped = mongolia.merge(soum_summary, left_on="Id", right_on="PARENT_ID", how="left")

# ==================== 5. Export source layer ====================
map_layer = mapped[["Id", "ADM1_NAME", "ADM3_NAME", "median_rho", "n_valid_years", "geometry"]].copy()
map_layer = map_layer.rename(columns={
    "Id": "ADMIN_ID",
    "ADM1_NAME": "AIMAG",
    "ADM3_NAME": "SOUM",
    "median_rho": "MED_RHO",
    "n_valid_years": "N_YEARS",
})

map_layer["EVALUATED"] = map_layer["MED_RHO"].notna().astype(np.int16)
map_layer["RHO_BIN"] = np.select(
    [
        map_layer["MED_RHO"] < 0,
        map_layer["MED_RHO"].between(0, 0.3, inclusive="left"),
        map_layer["MED_RHO"].between(0.3, 0.5, inclusive="left"),
        map_layer["MED_RHO"].between(0.5, 0.7, inclusive="left"),
        map_layer["MED_RHO"] >= 0.7,
    ],
    ["LT_0", "0_0.3", "0.3_0.5", "0.5_0.7", "GE_0.7"],
    default="NO_DATA",
)

map_layer["N_YEARS"] = map_layer["N_YEARS"].fillna(0).astype(np.int16)
map_layer.to_file(OUTPUT_DIR / "soum_median_ranking_2012_2023.shp", driver="ESRI Shapefile", encoding="UTF-8", index=False)

# ==================== 6. Classification ====================
bins = [-np.inf, 0.0, 0.3, 0.5, 0.7, np.inf]
labels = [r"$\rho < 0$", r"$0 \leq \rho < 0.3$", r"$0.3 \leq \rho < 0.5$", r"$0.5 \leq \rho < 0.7$", r"$\rho \geq 0.7$"]
legend_labels = [r"$<0$", r"$0$–$0.3$", r"$0.3$–$0.5$", r"$0.5$–$0.7$", r"$\geq0.7$"]

colors = ["#D04A3A", "#DCEAF4", "#B8D3E7", "#78A9CE", "#2F6F9F"]
NO_DATA_COLOR = "#D9D9D9"

mapped["rho_class"] = pd.cut(mapped["median_rho"], bins=bins, labels=labels, right=False)

# ==================== 7. Create figure ====================
fig = plt.figure(figsize=(FIG_W / 25.4, FIG_H / 25.4), constrained_layout=False)
ax = fig.add_axes(rect_mm(FIG_W, FIG_H, MAP_LEFT, BOTTOM, MAP_W, MAP_H))
ax_leg = fig.add_axes(rect_mm(FIG_W, FIG_H, LEG_LEFT, BOTTOM, LEG_W, LEG_H))
ax_leg.axis("off")

# ==================== 8. Draw map ====================
mongolia.plot(ax=ax, facecolor=NO_DATA_COLOR, edgecolor="white", linewidth=0.20, zorder=1)

for label, color in zip(labels, colors):
    subset = mapped.loc[mapped["rho_class"].astype("object").eq(label)]
    if not subset.empty:
        subset.plot(ax=ax, facecolor=color, edgecolor="white", linewidth=0.22, zorder=2)

mongolia.dissolve(by="ADM1_NAME").boundary.plot(ax=ax, color="#626262", linewidth=0.42, zorder=3)
mongolia.dissolve().boundary.plot(ax=ax, color="#292929", linewidth=0.70, zorder=4)

xmin_map, ymin_map, xmax_map, ymax_map = mongolia.total_bounds
xpad = (xmax_map - xmin_map) * 0.012
ypad = (ymax_map - ymin_map) * 0.012
ax.set_xlim(xmin_map - xpad, xmax_map + xpad)
ax.set_ylim(ymin_map - ypad, ymax_map + ypad)
ax.set_aspect("equal", adjustable="box")
ax.set_axis_off()

# ==================== 9. Panel title ====================
fig.canvas.draw()
map_bbox = ax.get_position()
TITLE_GAP_MM = 1.0

fig.text(
    TITLE_LEFT / FIG_W,
    map_bbox.y1 + TITLE_GAP_MM / FIG_H,
    "(c) Spatial distribution of within-soum ranking consistency",
    ha="left", va="bottom", fontsize=9, fontweight="normal"
)

# ==================== 10. Scale bar ====================
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

scale_length = 300000
scale_x = xmin + 0.055 * (xmax - xmin)
scale_y = ymin + 0.065 * (ymax - ymin)

ax.plot([scale_x, scale_x + scale_length], [scale_y, scale_y], color="#2F2F2F", lw=1.0)
ax.plot([scale_x, scale_x], [scale_y - 11000, scale_y + 11000], color="#2F2F2F", lw=0.5)
ax.plot([scale_x + scale_length, scale_x + scale_length], [scale_y - 11000, scale_y + 11000], color="#2F2F2F", lw=0.5)
ax.text(scale_x, scale_y + 22000, "0", ha="center", va="bottom", fontsize=8)
ax.text(scale_x + scale_length, scale_y + 22000, "300 km", ha="center", va="bottom", fontsize=8)

# ==================== 11. North arrow ====================
nx = 0.95
y0 = 0.80
y1 = 0.90

ax.plot([nx, nx], [y0, y1], transform=ax.transAxes, color="#2F2F2F", lw=0.75, solid_capstyle="butt", zorder=7)
ax.scatter([nx], [y1], transform=ax.transAxes, marker="^", s=34, facecolors="white", edgecolors="#2F2F2F", linewidths=0.8, zorder=8, clip_on=False)
ax.text(nx, y1 + 0.03, "N", transform=ax.transAxes, ha="center", va="bottom", fontsize=8, fontweight="normal", color="#2F2F2F")


# ==================== 12. Legend on right ====================
handles = [
    Line2D([0], [0], marker="s", linestyle="none", markersize=6.2,
           markerfacecolor=color, markeredgecolor="none", label=legend_label)
    for legend_label, color in zip(legend_labels, colors)
]
handles.append(
    Line2D([0], [0], marker="s", linestyle="none", markersize=6.2,
           markerfacecolor=NO_DATA_COLOR, markeredgecolor="#AAAAAA",
           markeredgewidth=0.35, label="Not evaluated")
)
ax_leg.legend(
    handles=handles,
    title=r"Median Spearman's $\rho$" + "\n" + r"(2012–2023)",
    loc="center left",
    bbox_to_anchor=(-0.50, 0.50),
    frameon=False,
    fontsize=7.2,
    title_fontsize=7.6,
    handletextpad=0.35,
    labelspacing=0.55,
    borderaxespad=0
)

# ==================== 13. Check physical dimensions ====================
fig.canvas.draw()
bbox = ax.get_position()
actual_left = bbox.x0 * FIG_W
actual_bottom = bbox.y0 * FIG_H
actual_width = bbox.width * FIG_W
actual_height = bbox.height * FIG_H

print(f"Figure size = {FIG_W:.2f} × {FIG_H:.2f} mm")
print(f"Map axes: left={actual_left:.2f} mm, bottom={actual_bottom:.2f} mm, width={actual_width:.2f} mm, height={actual_height:.2f} mm")

# ==================== 14. Save ====================

plt.savefig(OUTPUT_DIR / "Fig5_c_soum_median_ranking_map.png", dpi=1200, bbox_inches=None, facecolor="white")
plt.savefig(OUTPUT_DIR / "Fig5_c_soum_median_ranking_map.pdf", bbox_inches=None, facecolor="white")
#plt.savefig(OUTPUT_DIR / "Fig5_c_soum_median_ranking_map.svg", bbox_inches=None, facecolor="white")

plt.show()

In [ ]:
# ==================== Final Fig5: merge the fixed panels without redrawing ====================
from pathlib import Path
from PIL import Image
from IPython.display import display

AB_FIGURE = OUTPUT_DIR / "Fig5_ab_bagh_validation.png"
C_FIGURE = OUTPUT_DIR / "Fig5_c_soum_median_ranking_map.png"
FINAL_STEM = OUTPUT_DIR / "Fig5_bagh_cross_scale_validation_full_abc"

panel_ab = Image.open(AB_FIGURE).convert("RGB")
panel_c = Image.open(C_FIGURE).convert("RGB")

# Both finalized source figures are 127 mm wide at 1,200 dpi.
# A 2-mm white gap separates the two rows; neither source panel is resized or cropped.
if panel_ab.width != panel_c.width:
    raise ValueError(
        f"The finalized panel figures must have identical widths: "
        f"(a-b)={panel_ab.width}px, (c)={panel_c.width}px"
    )

gap_px = round(panel_ab.width * 2.0 / 127.0)
composite = Image.new(
    "RGB",
    (panel_ab.width, panel_ab.height + gap_px + panel_c.height),
    "white",
)
composite.paste(panel_ab, (0, 0))
composite.paste(panel_c, (0, panel_ab.height + gap_px))

composite.save(FINAL_STEM.with_suffix(".png"), dpi=(1200, 1200))
composite.save(FINAL_STEM.with_suffix(".tiff"), dpi=(1200, 1200), compression="tiff_lzw")
composite.save(FINAL_STEM.with_suffix(".pdf"), resolution=1200.0)

print(f"Source (a-b): {panel_ab.size[0]} x {panel_ab.size[1]} px")
print(f"Source (c): {panel_c.size[0]} x {panel_c.size[1]} px")
print(f"Final Fig5: {composite.size[0]} x {composite.size[1]} px")
print(f"Final physical size: 127.0 x {composite.height / 1200 * 25.4:.1f} mm")
display(composite)
